# grads-dict-accumulate-parents — faded example 3: Faded: accumulate never overwrites, always adds

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grads-dict-accumulate-parents`. Running the beacon reports progress on the `Backprop: grads dict accumulate parents` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grads dict accumulate parents` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grads-dict-accumulate-parents`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grads-dict-accumulate-parents"
DD_SUBTOPIC = "Backprop: grads dict accumulate parents"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The critical correctness property of gradient accumulation is that the function must NEVER overwrite an existing gradient entry — it must always add. If two backward paths reach the same parent, each produces a contribution that must be summed. A function that writes `grads[parent] = g` (overwrite) instead of `grads[parent] = grads.get(parent, 0) + g` (accumulate) silently drops all but the last contribution, breaking the chain rule for any shared node in the graph.

## Faded exercise 3

Complete `safe_accumulate`. The blank is the assignment that correctly handles BOTH the first-visit case (parent not in grads) AND the revisit case (parent already has a gradient), ensuring no previous contribution is discarded.

**Fill in:** The assignment into grads[parent] that seeds with zero on first visit and adds to existing value on revisit, using + not +=.

In [ ]:
import torch as t

t.manual_seed(0)

class Node:
    def __init__(self, n): self.name = n

def safe_accumulate(grads, parent, g):
    grads[parent] = grads.get(parent, 0) + g

# Exercise: same parent, two contributions
p = Node('x')
grads = {}
g1 = t.tensor([5.0, -1.0])
g2 = t.tensor([2.0, 3.0])
safe_accumulate(grads, p, g1)
safe_accumulate(grads, p, g2)
print(grads[p])  # [7.0, 2.0]


def _test():
    import torch as t

    class Node:
        def __init__(self, n): self.name = n

    # Three contributions to same parent
    p = Node('x')
    grads = {}
    gs = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0]), t.tensor([-1.0, 1.0])]
    for g in gs:
        safe_accumulate(grads, p, g)
    expected = sum(gs)
    assert t.allclose(grads[p], expected), f"Expected {expected}, got {grads[p]}"

    # Rebind check: old ref not mutated
    p2 = Node('y')
    grads2 = {}
    g_a = t.tensor([10.0, 20.0])
    safe_accumulate(grads2, p2, g_a)
    old_ref = grads2[p2]
    safe_accumulate(grads2, p2, t.tensor([1.0, 1.0]))
    assert t.allclose(old_ref, g_a), "Rebind violated: old reference was mutated by +="


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class Node:
    def __init__(self, n): self.name = n

def safe_accumulate(grads, parent, g):
    grads[parent] = grads.get(parent, 0) + g

# Exercise: same parent, two contributions
p = Node('x')
grads = {}
g1 = t.tensor([5.0, -1.0])
g2 = t.tensor([2.0, 3.0])
safe_accumulate(grads, p, g1)
safe_accumulate(grads, p, g2)
print(grads[p])  # [7.0, 2.0]
```
</details>